# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syed-Attique/FlyRank-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

Can a model rank pages whose search impressions fell within March 2026 better than a hand-written rule? Decision supported: which pages an editor opens first (a shortlist of about 50 per cycle). Proxy label: impressions Mar 16-31 below Mar 1-15.

In [1]:
import json, os
while not os.path.isdir("docs") and os.getcwd() != "/": os.chdir("..")
R = json.load(open("docs/results.json"))
print(f"{R['pages']:,} pages | {R['train_clients']} train clients, {R['test_clients']} test clients | base rate {R['base_all']}")

176,738 pages | 37 train clients, 10 test clients | base rate 0.377


## 2. Data

FlyRank ML Internship warehouse (gated), `fact_content_daily_performance` month=2026-03 plus `dim_content`; 176,738 pages after dropping missing position or CTR; 37 training and 10 test clients. Excluded: GA4 fields (4.2% of rows), product scores, label-source columns. No client names, URLs or hash IDs on the page.

In [2]:
print("Test pages:", f"{R['test_pages']:,}", "| base rate in test:", R["base_test"])
print("GA4 share of March rows:", R["ga4_share"])

Test pages: 38,428 | base rate in test: 0.403
GA4 share of March rows: 0.042


## 3. Methodology

Seven page-level features measured over the whole month; baseline = age 365+ days and 500+ impressions; logistic regression and random forest (300 trees, depth 8, seed 42); one client-grouped split; precision@20/50 next to the base rate. Leakage checks: label-source column removed, random vs grouped split, top feature dropped.

In [3]:
import pandas as pd
pd.DataFrame(R["split"], columns=["split","p@20","p@50","source"])

,split,p@20,p@50,source
0,Random row split,0.85,0.86,w06
1,Grouped by client,0.55,0.66,w06


## 4. Results (vs baseline)

On 10 held-out clients, precision@50: base rate 0.40, rule 0.10, logistic regression 0.44, random forest 0.66-0.68. A random row split gave 0.86. Observed and directional; one split, 50 pages.

In [4]:
pd.DataFrame(R["methods"], columns=["method","p@20","p@50","source"])

,method,p@20,p@50,source
0,Base rate (random order),0.403,0.403,w05
1,Baseline rule: age 365+ days and 500+ impressions,0.050,0.100,w05
2,Logistic regression,0.450,0.440,w05
3,Random forest,0.550,0.680,w05 (w06 rerun: 0.55 / 0.66)


## 5. Limitations

Features and label share the same 31 days, so this is not a forecast. Small test (10 clients). Proxy label is noisy for tiny pages. One month. Associations only; no causal claim about refreshes.

In [5]:
print("Limitation check - RF p@50: w05 0.68 vs w06 0.66; Wilson 95% interval for 0.66 on 50 pages ~ 0.52-0.78 (ignores client clustering).")

Limitation check - RF p@50: w05 0.68 vs w06 0.66; Wilson 95% interval for 0.66 on 50 pages ~ 0.52-0.78 (ignores client clustering).


## 6. Ranked recommendations

1) Model shortlist with human check. 2) CTR/snippet review on well-ranked low-CTR pages, largest impressions first (untested ordering). 3) Do not use 'old and visible' alone. 4) Re-test with pre-window features on a later month.

In [6]:
a = R["actions"]; tot = sum(a.values())
for k, v in a.items(): print(f"{k:20s} {v:>8,} ({v/tot:.1%})")

fix_ctr_snippet       104,091 (58.9%)
review_for_refresh      5,448 (3.1%)
no_action              67,199 (38.0%)


## 7. Artifacts the paper embeds

Every number comes from `docs/results.json` (each row names its source notebook). `work/build_paper.py` regenerates `docs/index.html` from it.

In [7]:
import subprocess
print(subprocess.run(["python", "work/build_paper.py"], capture_output=True, text=True).stdout)

wrote docs/index.html 11386 bytes



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
